In [1]:
import sys
sys.path.insert(0, r"e:\TalentLens")

import torch
import torch.nn as nn
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from src.data.dataset import ResumeJDDataset
from src.models.biencoder import BiEncoderClassifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cuda


In [2]:
train_dataset = ResumeJDDataset(r"e:\TalentLens\data\processed\train_encodings.pt")
test_dataset  = ResumeJDDataset(r"e:\TalentLens\data\processed\test_encodings.pt")

print("Train size:", len(train_dataset))
print("Test size: ", len(test_dataset))

# Verify one example looks correct
sample = train_dataset[0]
print("\nSample keys:", list(sample.keys()))
print("Label:", sample['label'])
print("Resume input_ids shape:", sample['resume_input_ids'].shape)

Train size: 6241
Test size:  1759

Sample keys: ['resume_input_ids', 'resume_attention_mask', 'jd_input_ids', 'jd_attention_mask', 'label']
Label: tensor(0)
Resume input_ids shape: torch.Size([512])


In [3]:
# batch_size=16 is safe for free-tier GPU with BERT
# shuffle=True for train — important so model doesn't learn order
# shuffle=False for test — order doesn't matter for evaluation

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print("Train batches:", len(train_loader))
print("Test batches: ", len(test_loader))

Train batches: 391
Test batches:  110


In [4]:
# Remember the 50/25/25 imbalance from Week 1 EDA
# class_weight='balanced' tells the loss function to penalize
# mistakes on minority classes more heavily

from sklearn.utils.class_weight import compute_class_weight

train_labels = train_dataset.labels.numpy()

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights:", class_weights)

Class weights: tensor([0.6619, 1.3370, 1.3491], device='cuda:0')


In [5]:
model = BiEncoderClassifier(
    model_name='bert-base-uncased',
    num_classes=3,
    dropout=0.1
).to(device)

# AdamW is the standard optimizer for transformer fine-tuning
# Small learning rate — large lr destroys pretrained weights
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

# Total training steps
EPOCHS       = 3
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)  # 10% warmup

# Linear schedule with warmup — standard for BERT fine-tuning
# lr increases during warmup then linearly decays to 0
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# CrossEntropyLoss with class weights to handle imbalance
criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Total steps: {total_steps}, Warmup steps: {warmup_steps}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model parameters: 109,491,459
Total steps: 1173, Warmup steps: 117


In [6]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, total_correct, total_examples = 0, 0, 0

    for batch in loader:
        # Move batch to device
        resume_input_ids      = batch['resume_input_ids'].to(device)
        resume_attention_mask = batch['resume_attention_mask'].to(device)
        jd_input_ids          = batch['jd_input_ids'].to(device)
        jd_attention_mask     = batch['jd_attention_mask'].to(device)
        labels                = batch['label'].to(device)

        # Forward pass
        optimizer.zero_grad()
        logits = model(
            resume_input_ids, resume_attention_mask,
            jd_input_ids, jd_attention_mask
        )

        # Compute loss and backpropagate
        loss = criterion(logits, labels)
        loss.backward()

        # Clip gradients — prevents exploding gradients during BERT fine-tuning
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        # Track metrics
        preds          = logits.argmax(dim=1)
        total_loss    += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total_examples+= labels.size(0)

    avg_loss = total_loss / total_examples
    accuracy = total_correct / total_examples
    return avg_loss, accuracy

In [7]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_examples = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            resume_input_ids      = batch['resume_input_ids'].to(device)
            resume_attention_mask = batch['resume_attention_mask'].to(device)
            jd_input_ids          = batch['jd_input_ids'].to(device)
            jd_attention_mask     = batch['jd_attention_mask'].to(device)
            labels                = batch['label'].to(device)

            logits = model(
                resume_input_ids, resume_attention_mask,
                jd_input_ids, jd_attention_mask
            )

            loss           = criterion(logits, labels)
            preds          = logits.argmax(dim=1)

            total_loss    += loss.item() * labels.size(0)
            total_correct += (preds == labels).sum().item()
            total_examples+= labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / total_examples
    accuracy = total_correct / total_examples
    return avg_loss, accuracy, all_preds, all_labels

In [ ]:
train_losses, train_accs = [], []
test_losses,  test_accs  = [], []
best_macro_f1 = 0

for epoch in range(EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"{'='*50}")

    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, criterion, device
    )
    test_loss, test_acc, test_preds, test_labels = evaluate(
        model, test_loader, criterion, device
    )

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)

    # Compute macro F1 for this epoch
    from sklearn.metrics import f1_score
    macro_f1 = f1_score(test_labels, test_preds, average='macro')

    print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f}")
    print(f"Test  loss: {test_loss:.4f} | Test  acc: {test_acc:.4f}")
    print(f"Macro F1  : {macro_f1:.4f}")

    # Save best model
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        torch.save(model.state_dict(), r"e:\TalentLens\models\best_biencoder.pt")
        print(f"*** New best model saved (macro F1: {macro_f1:.4f}) ***")

print(f"\nBest macro F1: {best_macro_f1:.4f}")


Epoch 1/3


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, EPOCHS+1), train_losses, label='Train Loss', marker='o')
axes[0].plot(range(1, EPOCHS+1), test_losses,  label='Test Loss',  marker='o')
axes[0].set_title("Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(range(1, EPOCHS+1), train_accs, label='Train Accuracy', marker='o')
axes[1].plot(range(1, EPOCHS+1), test_accs,  label='Test Accuracy',  marker='o')
axes[1].set_title("Accuracy Curve")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig(r"e:\TalentLens\reports\figures\training_curves.png", dpi=150)
plt.show()

In [ ]:
# Load best saved model and evaluate
model.load_state_dict(torch.load(r"e:\TalentLens\models\best_biencoder.pt"))
_, _, final_preds, final_labels = evaluate(model, test_loader, criterion, device)

print("=== Final Evaluation — Best Bi-Encoder ===\n")
print(classification_report(
    final_labels,
    final_preds,
    target_names=['No Fit', 'Potential Fit', 'Good Fit']
))

# Confusion matrix
cm = confusion_matrix(final_labels, final_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['No Fit', 'Potential Fit', 'Good Fit'],
    yticklabels=['No Fit', 'Potential Fit', 'Good Fit']
)
plt.title("Fine-tuned Bi-Encoder — Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(r"e:\TalentLens\reports\figures\biencoder_confusion_matrix.png", dpi=150)
plt.show()

In [1]:
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

NVIDIA GeForce MX450
VRAM: 2.0 GB
